In [48]:
import torch
import pandas as pd
import torch.nn as nn
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [53]:
class CustomRunsDataset():
    def __init__(self, split='train', device='cpu',  context_len=10, file=3):
        # Carregar as colunas diretamente
        # df = pd.read_parquet(r'/home/mgteus/workspace/neuro/transformers_andrej/train_runs_15_100.gzip', columns=['x_pos', 'y_pos'])
        # df = pd.read_parquet(r'/home/mgteus/workspace/neuro/transformers_andrej/train_runs_15.gzip', columns=['x_pos', 'y_pos'])
        path = {0:r'/home/mgteus/workspace/navitrans/df_trainset_v0.csv',}
        
        df = pd.read_csv(path[file], usecols=['encode_position_state'])
        converted_arrays = []
        # Vectorização de arredondamento e conversão para tensor
        for val in df['encode_position_state'].values:
            # Usando regex para remover colchetes e quebras de linha, depois convertendo para array NumPy
            filtered_array = np.fromstring(re.sub(r'[\[\]\n]', ' ', val), sep=' ')
            converted_arrays.append(filtered_array)

        # Convertendo a lista de arrays em um único array NumPy (mantendo formato original)
        self.feature_array = np.array(converted_arrays, dtype=np.float32)
        # self.feature_array = np.array([np.array(val) for val in df['encode_position_state'].values])
        self.device = device
        # Convertendo para tensor de float32
        self.feature_array = torch.tensor(self.feature_array,)
        
        # Tamanho de treinamento e teste
        n = np.min([int(len(df) * 0.8), int(100_000)])
        
        # Dados para treino ou teste
        self.data = self.feature_array[:n] if split == 'train' else self.feature_array[n:]
        print('dataset sent to', device, 'with size ', len(self.data))
        self.data = self.data.to(device,)
        self.context_len = context_len

    # def __len__(self):
    #     # print(len(self.data))
    #     return len(self.data) - self.context_len

    # def __getitem__(self, idx):
    #     # Preparando o índice de entrada e saída
    #     x = self.data[idx: idx + self.context_len]
    #     y = self.data[idx + 1: idx + self.context_len + 1]

    #     # y = y.to(self.device)
    #     # x = x.to(self.device)
        
    #     return x, y


In [54]:
def custom_dataloader(dataset, batch_size, device):
        # Verifique o tamanho total do dataset
    dataset_size = len(dataset.data)
    # print('dataset is on ', dataset.data.get_device())
    context_len = dataset.context_len
    max_iter = dataset_size // batch_size
    print(max_iter, 'expected iters with size', batch_size)
    iter_counter = 0
    # Garante que índices aleatórios não ultrapassem os limites do dataset
    max_start_idx = dataset_size - context_len - 1
    def generator():
        for _ in range(max_iter): 
            # Gere n índices aleatórios no intervalo permitido
            ix = torch.randint(len(dataset.data) - context_len -1, (batch_size,))
            x = torch.stack([dataset.data[i:i+context_len] for i in ix])
            y = torch.stack([dataset.data[i+1:i+context_len+1] for i in ix])

            # x, y = x.to(device), y.to(device)

            yield x, y
        
    return generator()

In [67]:
class Head(nn.Module):
    def __init__(self, context_len, batch_size, dropout, head_size, output_dim) -> None:
        super().__init__()
        # parameters
        self.batch_size = batch_size
        self.context_len = context_len
        self.dropout_value = dropout
        self.head_size = head_size
        self.output_dim = output_dim

        # layers
        #   # static layers
        # self.pos_to_enc_layer = nn.Linear(2, 2,)
        self.enc_layer = nn.Linear(2, 1)
        self.output_layer = nn.Linear(self.head_size, self.output_dim, bias=False)
        #   # dynamic layers
        self.key = nn.Linear(self.head_size, self.head_size, bias=False)
        self.query = nn.Linear(self.head_size, self.head_size, bias=False)
        self.values = nn.Linear(self.head_size, self.head_size, bias=False)

        # tril
        self.register_buffer('tril', torch.tril(torch.ones(self.context_len, self.context_len)))

        # dropout
        self.dropout = nn.Dropout(self.dropout_value)


    def forward(self, x):
        # x = self.pos_to_enc_layer(x) # [x, y] -> [i, j]
        # x = self.enc_layer(x).squeeze(-1)        # [i, j] -> [k]

        B, C, _ = x.shape
        if C < self.context_len:
            pad_amount = self.context_len - C
            # Padding na segunda dimensão (context_len)
            x = nn.functional.pad(x, (0, 0, 0, pad_amount), mode="constant", value=0)

        k = self.key(x)
        q =  self.query(x) #self.key(x)

        wei = q @ k.transpose(-2, -1) * self.head_size**-0.5 # [B, C] @ [C, B] -> [B, B]
        # wei = wei.masked_fill(self.tril[:C, :C] == 0, float('-inf'))
        # wei = nn.functional.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        # print(wei.var(),)
        v = self.values(x)
        out = wei @ v # [B, B] @ [B, C] -> [B, C]
        
        out = self.output_layer(out) # [B, C] -> [B, 2]

        return out

In [68]:
CONTEXT_LEN = 32
BATCH_SIZE = 64
DROPOUT = 0.1
LEARNING_RATE = 1e-5
NUM_HEADS = 2
NUM_EPOCHS = 1e1
NUM_BLOCKS = 2
HEAD_SIZE = 21
HEAD_OUTPUT_DIM = HEAD_SIZE // NUM_HEADS
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [61]:
dataset_train = CustomRunsDataset(split='train', device=DEVICE, context_len=CONTEXT_LEN, file=0)
# dataset_test = CustomRunsDataset(split='test', device=DEVICE, context_len=CONTEXT_LEN, file=2)

dataloader = custom_dataloader(dataset=dataset_train, batch_size=BATCH_SIZE, device=DEVICE)

dataset sent to cpu with size  6952
108 expected iters with size 64


In [62]:
head = Head(
    context_len=CONTEXT_LEN,
    batch_size=BATCH_SIZE,
    dropout=DROPOUT,
    head_size=HEAD_SIZE,
    output_dim=HEAD_OUTPUT_DIM
)

In [63]:
for x, y in dataloader:
    break

In [65]:
x.shape

torch.Size([64, 32, 21])

In [70]:
key = nn.Linear(21, 21)

In [74]:
x0 = x[0:1, :, :]
x0.shape

torch.Size([1, 32, 21])

In [75]:
head(x0)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x21 and 2x21)